# Understanding Sentence Embeddings

In this notebook, we explore how to prepare our Word2Vec embeddings for machine learning models by representing full sentences (or tickets) instead of just single words. 

We will look at two approaches:
1. **Average Embeddings** (Good for traditional ML like Logistic Regression / SVM)
2. **Sequence Embeddings with Padding** (Required for Deep Learning like RNNs / LSTMs)

In [ ]:
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load our custom trained Skip-Gram Word2Vec model
model_path = "../artifacts/word2vec/skipgram_v100.model"
w2v_model = Word2Vec.load(model_path)

print(f"Word2Vec model loaded successfully! Vocabulary size: {len(w2v_model.wv)}")

## 1. Average Embeddings

We generate one single vector for the entire ticket by averaging the Word2Vec embeddings of all its words. If our embedding size is 100, the resulting vector for the entire sentence will also be exactly 100.

In [ ]:
tokens = ["payment", "failed", "refund"]

# Extract vectors for words that exist in the vocabulary
vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]

if len(vectors) > 0:
    # Average across all tokens to create a single Document Vector
    avg_embedding = np.mean(vectors, axis=0)
    
    print(f"Tokens: {tokens}")
    print(f"Average Embedding Shape: {avg_embedding.shape}") # Expected: (100,)
else:
    print("None of the words were found in the vocabulary.")

## 2. Sequence Embeddings (For Deep Learning)

Instead of crushing the sequence into one average vector, we keep the temporal order of words. We generate a 2D matrix of shape `(number_of_tokens, embedding_dimension)`.

Because neural networks require fixed-size inputs, we then pad (or truncate) the sequences to a `maxlen` using Keras `pad_sequences`.

In [ ]:
# Example 15-word ticket
ticket_tokens = ["i", "tried", "to", "make", "a", "payment", "but", "the", "system", "failed", "so", "i", "want", "a", "refund"]
print(f"Original ticket length: {len(ticket_tokens)} words")

# Convert tokens to a sequence of vectors
sequence = [w2v_model.wv[word] for word in ticket_tokens if word in w2v_model.wv]
sequence = np.array(sequence)

print(f"\nSequence Shape before padding: {sequence.shape}")
print("(Expected something close to (15, 100) depending on vocabulary presence)")

# Define fixed length for our neural network input
MAX_SEQUENCE_LENGTH = 20

# pad_sequences expects a list of sequences, so we wrap our sequence in a list: [sequence]
# We use dtype='float32' because we are padding multidimensional float arrays, not integers.
padded_sequences = pad_sequences(
    [sequence], 
    maxlen=MAX_SEQUENCE_LENGTH, 
    padding='post',    # Add zeros at the end if too short
    truncating='post', # Cut off at the end if too long
    dtype='float32'
)

# Extract the first (and only) padded item
padded_sequence = padded_sequences[0]

print(f"\nPadded Sequence Shape: {padded_sequence.shape}")
print("(Expected exactly (20, 100))")

print("\nNow the data is perfectly shaped to be fed directly into an LSTM/BiLSTM layer!")